# Coco Crepe — 08 Validate Data Products

Validaciones técnicas y funcionales para demostrar que las reglas de limpieza, deduplicación y reconciliación fueron aplicadas correctamente.

In [0]:
GROUP = "g203"

SALES_DATA_PRODUCT = "sales_summary"
INVENTORY_DATA_PRODUCT = "inventory_status"
PRODUCT_DATA_PRODUCT = "product_master"

# Completa estos valores solo si la detección automática no encuentra
# exactamente un catálogo por Data Product.
SALES_CATALOG_MANUAL = None
INVENTORY_CATALOG_MANUAL = "g203_inv_inventory_status"
PRODUCT_CATALOG_MANUAL = None

def resolve_catalog(data_product_name, manual_catalog=None):
    if manual_catalog:
        return manual_catalog

    catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
    target = data_product_name.lower()

    preferred = [
        catalog for catalog in catalogs
        if GROUP.lower() in catalog.lower()
        and target in catalog.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    matches = [
        catalog for catalog in catalogs
        if target in catalog.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"No se pudo identificar un catálogo único para '{data_product_name}'. "
        f"Catálogos visibles: {catalogs}. "
        "Completa la variable *_CATALOG_MANUAL correspondiente."
    )

SALES_CATALOG = resolve_catalog(
    SALES_DATA_PRODUCT,
    SALES_CATALOG_MANUAL
)

INVENTORY_CATALOG = resolve_catalog(
    INVENTORY_DATA_PRODUCT,
    INVENTORY_CATALOG_MANUAL
)

PRODUCT_CATALOG = resolve_catalog(
    PRODUCT_DATA_PRODUCT,
    PRODUCT_CATALOG_MANUAL
)

print(f"Sales catalog: {SALES_CATALOG}")
print(f"Inventory catalog: {INVENTORY_CATALOG}")
print(f"Product catalog: {PRODUCT_CATALOG}")

## 1. Existencia de tablas

In [0]:
spark.sql(f"SHOW TABLES IN {PRODUCT_CATALOG}.bronze").display()
spark.sql(f"SHOW TABLES IN {PRODUCT_CATALOG}.silver").display()
spark.sql(f"SHOW TABLES IN {PRODUCT_CATALOG}.gold").display()

spark.sql(f"SHOW TABLES IN {SALES_CATALOG}.bronze").display()
spark.sql(f"SHOW TABLES IN {SALES_CATALOG}.silver").display()
spark.sql(f"SHOW TABLES IN {SALES_CATALOG}.gold").display()

spark.sql(f"SHOW TABLES IN {INVENTORY_CATALOG}.bronze").display()
spark.sql(f"SHOW TABLES IN {INVENTORY_CATALOG}.silver").display()
spark.sql(f"SHOW TABLES IN {INVENTORY_CATALOG}.gold").display()

## 2. Evidencia de deduplicación: Bronze vs Silver

In [0]:
spark.sql(f"""
SELECT
    'products' AS dataset,
    (SELECT COUNT(*) FROM {PRODUCT_CATALOG}.bronze.products) AS bronze_rows,
    (SELECT COUNT(*) FROM {PRODUCT_CATALOG}.silver.products_clean) AS silver_rows,
    (
        SELECT COUNT(*) - COUNT(DISTINCT product_id)
        FROM {PRODUCT_CATALOG}.bronze.products
    ) AS bronze_duplicate_keys,
    (
        SELECT COUNT(*) - COUNT(DISTINCT product_id)
        FROM {PRODUCT_CATALOG}.silver.products_clean
    ) AS silver_duplicate_keys

UNION ALL

SELECT
    'order_items',
    (SELECT COUNT(*) FROM {SALES_CATALOG}.bronze.order_items),
    (SELECT COUNT(*) FROM {SALES_CATALOG}.silver.sales_detail),
    (
        SELECT COUNT(*) - COUNT(DISTINCT item_id)
        FROM {SALES_CATALOG}.bronze.order_items
    ),
    (
        SELECT COUNT(*) - COUNT(DISTINCT item_id)
        FROM {SALES_CATALOG}.silver.sales_detail
    )

UNION ALL

SELECT
    'inventory',
    (SELECT COUNT(*) FROM {INVENTORY_CATALOG}.bronze.inventory),
    (SELECT COUNT(*) FROM {INVENTORY_CATALOG}.silver.inventory_detail),
    (
        SELECT COUNT(*) - COUNT(DISTINCT product_id)
        FROM {INVENTORY_CATALOG}.bronze.inventory
    ),
    (
        SELECT COUNT(*) - COUNT(DISTINCT product_id)
        FROM {INVENTORY_CATALOG}.silver.inventory_detail
    )
""").display()

## 3. Calidad de Gold Product Master

In [0]:
spark.sql(f"""
SELECT
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_ids,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_ids,
    SUM(
        CASE
            WHEN product_name IS NULL OR TRIM(product_name) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_names,
    SUM(
        CASE
            WHEN category IS NULL OR TRIM(category) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_categories,
    SUM(CASE WHEN price IS NULL OR price <= 0 THEN 1 ELSE 0 END)
        AS invalid_prices,
    SUM(CASE WHEN is_active IS NULL THEN 1 ELSE 0 END)
        AS null_active_status
FROM {PRODUCT_CATALOG}.gold.product_master
""").display()

## 4. Integridad referencial de ventas

In [0]:
spark.sql(f"""
SELECT COUNT(*) AS order_items_without_order
FROM {SALES_CATALOG}.bronze.order_items oi
LEFT JOIN {SALES_CATALOG}.bronze.orders o
    ON oi.order_id = o.order_id
WHERE o.order_id IS NULL
""").display()

spark.sql(f"""
SELECT COUNT(*) AS order_items_without_product
FROM {SALES_CATALOG}.bronze.order_items oi
LEFT JOIN {PRODUCT_CATALOG}.gold.product_master p
    ON oi.product_id = p.product_id
WHERE p.product_id IS NULL
""").display()

## 5. Calidad de Silver Sales

In [0]:
spark.sql(f"""
SELECT
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_ids,
    SUM(CASE WHEN item_id IS NULL THEN 1 ELSE 0 END) AS null_item_ids,
    COUNT(*) - COUNT(DISTINCT item_id) AS duplicate_items,
    SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) AS invalid_quantities,
    SUM(CASE WHEN unit_price <= 0 THEN 1 ELSE 0 END) AS invalid_prices,
    SUM(
        CASE
            WHEN ABS(line_total - ROUND(quantity * unit_price, 2)) > 0.01
            THEN 1 ELSE 0
        END
    ) AS incorrect_line_totals
FROM {SALES_CATALOG}.silver.sales_detail
""").display()

## 6. Reconciliación Silver vs Gold Sales

In [0]:
spark.sql(f"""
WITH silver_summary AS (
    SELECT
        order_date,
        COUNT(DISTINCT order_id) AS expected_orders,
        COUNT(DISTINCT customer_id) AS expected_customers,
        SUM(quantity) AS expected_units,
        ROUND(SUM(line_total), 2) AS expected_revenue
    FROM {SALES_CATALOG}.silver.sales_detail
    GROUP BY order_date
)
SELECT
    g.order_date,
    g.total_orders,
    s.expected_orders,
    g.total_orders - s.expected_orders AS orders_difference,
    g.unique_customers,
    s.expected_customers,
    g.unique_customers - s.expected_customers AS customers_difference,
    g.units_sold,
    s.expected_units,
    g.units_sold - s.expected_units AS units_difference,
    g.total_revenue,
    s.expected_revenue,
    ROUND(g.total_revenue - s.expected_revenue, 2) AS revenue_difference
FROM {SALES_CATALOG}.gold.sales_summary g
INNER JOIN silver_summary s
    ON g.order_date = s.order_date
ORDER BY g.order_date
""").display()

## 7. Validar ticket promedio

In [0]:
spark.sql(f"""
SELECT
    order_date,
    total_orders,
    total_revenue,
    average_ticket,
    ROUND(total_revenue / total_orders, 2) AS expected_average_ticket,
    ROUND(
        average_ticket - ROUND(total_revenue / total_orders, 2),
        2
    ) AS difference
FROM {SALES_CATALOG}.gold.sales_summary
WHERE ABS(
    average_ticket - ROUND(total_revenue / total_orders, 2)
) > 0.01
""").display()

## 8. Integridad referencial y calidad de inventario

In [0]:
spark.sql(f"""
SELECT COUNT(*) AS inventory_without_product
FROM {INVENTORY_CATALOG}.bronze.inventory i
LEFT JOIN {PRODUCT_CATALOG}.gold.product_master p
    ON i.product_id = p.product_id
WHERE p.product_id IS NULL
""").display()

spark.sql(f"""
SELECT
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_ids,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_products,
    SUM(CASE WHEN stock < 0 THEN 1 ELSE 0 END) AS negative_stock,
    SUM(CASE WHEN product_price <= 0 THEN 1 ELSE 0 END) AS invalid_prices,
    SUM(CASE WHEN supplier_id IS NULL THEN 1 ELSE 0 END) AS null_supplier_ids,
    SUM(CASE WHEN supplier_name IS NULL THEN 1 ELSE 0 END) AS null_supplier_names
FROM {INVENTORY_CATALOG}.silver.inventory_detail
""").display()

## 9. Validar Gold Inventory Status

In [0]:
spark.sql(f"""
SELECT *
FROM {INVENTORY_CATALOG}.gold.inventory_status
WHERE stock_status <>
    CASE
        WHEN stock < 50 THEN 'LOW'
        WHEN stock <= 150 THEN 'MEDIUM'
        ELSE 'OK'
    END
""").display()

spark.sql(f"""
SELECT
    product_id,
    stock,
    product_price,
    inventory_value,
    ROUND(stock * product_price, 2) AS expected_inventory_value
FROM {INVENTORY_CATALOG}.gold.inventory_status
WHERE ABS(
    inventory_value - ROUND(stock * product_price, 2)
) > 0.01
""").display()

## 10. Resumen final de Data Products

In [0]:
spark.sql(f"""
SELECT
    'product_master' AS data_product,
    COUNT(*) AS records
FROM {PRODUCT_CATALOG}.gold.product_master

UNION ALL

SELECT
    'sales_summary',
    COUNT(*)
FROM {SALES_CATALOG}.gold.sales_summary

UNION ALL

SELECT
    'inventory_status',
    COUNT(*)
FROM {INVENTORY_CATALOG}.gold.inventory_status
""").display()